# PY100 - Environment & Tools Setup
### A practical guide to Python environments, VS Code, Jupyter Notebooks, and Git

Before writing a single line of Python, you need a **reproducible environment** and the right **tooling**. This notebook covers the full setup — from virtual environments to version control.

## Topics Covered
1. Why Virtual Environments Matter
2. Creating Environments — `pip/venv`, `uv`, and `conda`
3. Kernel & Git-Related CLIs
4. VS Code — Essential Shortcuts
5. Jupyter Notebook — Files, Kernels, Markdown vs Code
6. Git & Git Bash — CLI Reference

> **Perspective:** A project without an isolated environment is a ticking time bomb. One library upgrade in your global Python can silently break every project on your machine. Virtual environments give each project its own sandbox — its own Python version, its own packages, its own rules.

---
## 1. Why Virtual Environments Matter

Every Python project depends on **external packages** (e.g., `pandas`, `requests`, `flask`). Without isolation:

| Problem | Example |
|---|---|
| **Version conflicts** | Project A needs `numpy==1.24`, Project B needs `numpy==2.0` |
| **Global pollution** | Installing packages globally clutters your system Python |
| **Reproducibility** | A colleague clones your repo — which versions should they install? |
| **Deployment risk** | "Works on my machine" — fails in production because of version drift |

A **virtual environment** creates an isolated directory containing:
- A copy (or symlink) of the Python interpreter
- Its own `site-packages` folder for installed libraries
- Activation scripts that temporarily modify your `PATH`

> **Rule of thumb:** One project = one virtual environment. Always.

---
## 2. Creating Environments — Three Methods

### Method 1: `pip` + `venv` (Built-in, No Extra Install)

`venv` ships with Python 3.3+ — no installation required. `pip` is the default package installer.

#### Step-by-step

```bash
# 1. Create the virtual environment
python -m venv .venv

# 2. Activate it
#    Windows (CMD)
.venv\Scripts\activate
#    Windows (PowerShell)
.venv\Scripts\Activate.ps1
#    Windows (Git Bash) / macOS / Linux
source .venv/Scripts/activate      # Git Bash on Windows
source .venv/bin/activate           # macOS / Linux

# 3. Verify — should point to .venv
which python        # Git Bash / macOS / Linux
where python         # Windows CMD

# 4. Install packages
pip install requests pandas flask

# 5. Freeze dependencies to a file
pip freeze > requirements.txt

# 6. Recreate the environment elsewhere
pip install -r requirements.txt

# 7. Deactivate when done
deactivate
```

**Key files:**
- `requirements.txt` — flat list of packages with pinned versions

> **Tip:** Add `.venv/` to your `.gitignore` — never commit the virtual environment folder.

---
### Method 2: `uv` (Modern, Fast, Rust-based)

`uv` is a next-generation Python package manager written in Rust. It is **dramatically faster** than `pip` (10–100x), handles virtual environments, and replaces `pip`, `pip-tools`, `venv`, and even `pyenv` in a single tool.

#### Installation

```bash
# Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Or via pip (if you already have Python)
pip install uv
```

#### Step-by-step

```bash
# 1. Initialize a new project (creates pyproject.toml)
uv init my-project
cd my-project

# 2. Create virtual environment (defaults to .venv)
uv venv

# 3. Activate it (same as venv — uv creates a standard .venv)
source .venv/Scripts/activate      # Git Bash on Windows
source .venv/bin/activate           # macOS / Linux

# 4. Add packages (automatically installs + updates pyproject.toml + lock file)
uv add requests pandas flask

# 5. Remove a package
uv remove flask

# 6. Sync environment from lock file (like pip install -r but deterministic)
uv sync

# 7. Run a script without manual activation
uv run python my_script.py

# 8. Install a specific Python version (replaces pyenv)
uv python install 3.12

# 9. Deactivate
deactivate
```

**Key files:**
- `pyproject.toml` — project metadata and dependency declarations
- `uv.lock` — deterministic lock file (commit this to Git)

> **Why uv?** Speed is the headline, but the real benefit is **unification** — one tool replaces `pip`, `venv`, `pip-tools`, `pyenv`, and `pipx`. It also generates a lock file automatically, ensuring deterministic installs across machines.

---
### Method 3: `conda` (Data Science & Cross-language)

`conda` is both a **package manager** and an **environment manager**. It is popular in data science because it can install non-Python dependencies (C libraries, R, CUDA toolkits) that `pip` cannot.

#### Installation

Choose one:
- **Miniconda** (lightweight, recommended): https://docs.conda.io/en/latest/miniconda.html
- **Anaconda** (full distribution with 250+ packages pre-installed): https://www.anaconda.com/download

#### Step-by-step

```bash
# 1. Create an environment with a specific Python version
conda create --name myenv python=3.12

# 2. Activate it
conda activate myenv

# 3. Install packages
conda install numpy pandas scikit-learn

# 4. Install from pip (for packages not on conda channels)
pip install some-package-not-on-conda

# 5. List all packages in the environment
conda list

# 6. Export environment to a YAML file
conda env export > environment.yml

# 7. Recreate from YAML on another machine
conda env create -f environment.yml

# 8. List all environments
conda env list

# 9. Deactivate
conda deactivate

# 10. Remove an environment entirely
conda env remove --name myenv
```

**Key files:**
- `environment.yml` — conda environment specification

> **When to use conda:** If your project depends on compiled scientific libraries (NumPy with MKL, TensorFlow with CUDA, R packages), conda handles the binary dependencies that pip cannot. For pure Python projects, `uv` or `pip/venv` is simpler.

---
### Comparison at a Glance

| Feature | `pip` + `venv` | `uv` | `conda` |
|---|---|---|---|
| Built into Python | Yes | No (install separately) | No (install separately) |
| Speed | Moderate | Very fast (Rust) | Slow (solver overhead) |
| Lock file | Manual (`pip freeze`) | Automatic (`uv.lock`) | Manual (`environment.yml`) |
| Non-Python deps | No | No | Yes (C, R, CUDA, etc.) |
| Python version mgmt | No (need `pyenv`) | Yes (`uv python install`) | Yes (`python=3.12`) |
| Config file | `requirements.txt` | `pyproject.toml` | `environment.yml` |
| Best for | Simple projects | Modern Python projects | Data science / ML |

---
## 3. Kernel & Git-Related CLIs

### Jupyter Kernel CLI (`ipykernel`)

A **kernel** is the compute engine that runs your notebook code. Each virtual environment can be registered as its own kernel.

```bash
# Install ipykernel inside your active virtual environment
pip install ipykernel

# Register the current environment as a Jupyter kernel
python -m ipykernel install --user --name=myenv --display-name "My Project (Python 3.12)"

# List all installed kernels
jupyter kernelspec list

# Remove a kernel
jupyter kernelspec uninstall myenv
```

> **Why register a kernel?** If you open a notebook in VS Code or Jupyter Lab, you need to select which Python environment runs the code. Without a registered kernel, your environment's packages will not be available.

### Git CLI (quick check)

```bash
# Check if git is installed
git --version

# Configure identity (one-time setup)
git config --global user.name "Your Name"
git config --global user.email "you@example.com"

# See current config
git config --list
```

### Other Useful CLIs

| CLI | Purpose | Install |
|---|---|---|
| `jupyter` | Launch Jupyter Lab / Notebook | `pip install jupyterlab` |
| `ipython` | Enhanced interactive Python shell | `pip install ipython` |
| `gh` | GitHub CLI (PRs, issues, repos) | https://cli.github.com |
| `pyenv` | Manage multiple Python versions | https://github.com/pyenv/pyenv |
| `pipx` | Install CLI tools in isolated envs | `pip install pipx` |
| `pre-commit` | Git hook framework for linting | `pip install pre-commit` |

---
## 4. VS Code — Essential Shortcuts (Windows)

### General Navigation

| Shortcut | Action |
|---|---|
| `Ctrl + P` | Quick Open — search and open any file by name |
| `Ctrl + Shift + P` | Command Palette — run any VS Code command |
| `Ctrl + B` | Toggle the sidebar (Explorer panel) |
| `Ctrl + J` | Toggle the bottom panel (Terminal, Output, Problems) |
| `Ctrl + \` | Split the editor (side by side) |
| `Ctrl + Tab` | Switch between open files |
| `Ctrl + W` | Close the current tab |
| `Ctrl + Shift + E` | Focus the Explorer panel |
| `Ctrl + Shift + F` | Search across all files in the workspace |
| `Ctrl + Shift + H` | Search and replace across all files |
| `Ctrl + ,` | Open Settings |

### Editing & Code

| Shortcut | Action |
|---|---|
| `Ctrl + /` | Toggle line comment (works in Python, JS, etc.) |
| `Shift + Alt + A` | Toggle block comment (`""" ... """` or `/* ... */`) |
| `Alt + Up/Down` | Move the current line up or down |
| `Shift + Alt + Up/Down` | Duplicate the current line up or down |
| `Ctrl + D` | Select the next occurrence of the current word |
| `Ctrl + Shift + L` | Select **all** occurrences of the current word |
| `Ctrl + L` | Select the entire current line |
| `Ctrl + Shift + K` | Delete the entire current line |
| `Ctrl + Enter` | Insert a blank line below (without breaking current line) |
| `Ctrl + Shift + Enter` | Insert a blank line above |
| `Ctrl + ]` | Indent the current line |
| `Ctrl + [` | Outdent the current line |
| `Ctrl + Z` | Undo |
| `Ctrl + Y` | Redo |
| `Ctrl + Shift + \` | Jump to the matching bracket |
| `F2` | Rename a symbol (variable, function) across all files |
| `F12` | Go to definition |
| `Alt + F12` | Peek definition (inline popup) |
| `Shift + F12` | Find all references |

### Terminal

| Shortcut | Action |
|---|---|
| `` Ctrl + ` `` | Toggle the integrated terminal |
| `` Ctrl + Shift + ` `` | Create a new terminal instance |
| `Ctrl + Shift + C` | Open an external terminal at workspace root |

### Multi-cursor & Selection

| Shortcut | Action |
|---|---|
| `Alt + Click` | Add a cursor at the click position |
| `Ctrl + Alt + Up/Down` | Add a cursor above / below |
| `Ctrl + Shift + L` | Add cursors at every occurrence of the selection |
| `Shift + Alt + drag` | Column (box) select |

### Useful Commands (Command Palette)

| Command | What it does |
|---|---|
| `Python: Select Interpreter` | Choose which Python / venv to use |
| `Python: Create Terminal` | Open a terminal with the selected venv activated |
| `Notebook: Select Kernel` | Pick a kernel for the current notebook |
| `Format Document` | Auto-format the current file (requires a formatter like Black) |
| `Toggle Word Wrap` | Wrap long lines in the editor |
| `Preferences: Color Theme` | Switch color themes |

> **Tip:** You do not need to memorize all shortcuts at once. Start with `Ctrl + P` (open file), `Ctrl + Shift + P` (command palette), `Ctrl + /` (comment), and `Alt + Up/Down` (move line). These four will cover 80% of your daily workflow.

---
## 5. Jupyter Notebook — Files, Kernels, Markdown vs Code

### File Extension & Naming

- Jupyter notebooks use the **`.ipynb`** extension (stands for **I**nteractive **Py**thon **N**ote**b**ook)
- Internally, `.ipynb` files are **JSON** — each cell's source, outputs, and metadata are stored as JSON objects
- **Naming convention:** Use descriptive names with underscores or hyphens:
  - `01_Basics.ipynb` — numbered prefix for ordering
  - `data_analysis_report.ipynb` — descriptive for project notebooks
  - Avoid spaces in filenames if possible (they work but cause issues in terminal commands)

### Kernel Installation & Selection

A **kernel** is the process that actually executes the code in your notebook cells.

```bash
# Step 1: Activate your virtual environment
source .venv/Scripts/activate     # Git Bash on Windows

# Step 2: Install ipykernel
pip install ipykernel

# Step 3: Register the kernel
python -m ipykernel install --user --name=python-demo --display-name "Python Demo"

# Step 4: In VS Code — click "Select Kernel" (top-right of notebook) → pick your kernel
```

> **Important:** If you install a package with `pip install pandas` but your notebook uses a different kernel, the import will fail. Always make sure the notebook kernel matches the environment where your packages are installed.

### Markdown Cells vs Code Cells

| Feature | Markdown Cell | Code Cell |
|---|---|---|
| **Purpose** | Documentation, headings, explanations | Executable Python code |
| **Execution** | Renders formatted text | Runs code and shows output below |
| **Supports** | Headings, bold, tables, links, images, LaTeX | Python (or any kernel language) |
| **Output** | Rendered HTML | Text, tables, plots, errors |
| **When to use** | Explain *why* and *what*, section headers | Write and run actual code |

**Why markdown cells matter:**
- They turn a notebook from a script into a **narrative document**
- Future you (and your teammates) will thank you for explaining the reasoning behind code
- Well-structured markdown (headings, lists) makes notebooks **navigable** via the outline panel
- Use markdown cells to break your notebook into logical sections — just like chapters in a book

### Jupyter Shortcuts in VS Code (Windows)

**Cell Navigation & Execution:**

| Shortcut | Action |
|---|---|
| `Ctrl + Enter` | Run the current cell (stay on it) |
| `Shift + Enter` | Run the current cell and move to the next |
| `Alt + Enter` | Run the current cell and insert a new cell below |
| `Ctrl + Shift + Enter` | Run all cells in the notebook |
| `Up / Down` | Navigate between cells (when not editing) |
| `Enter` | Enter edit mode in the selected cell |
| `Escape` | Exit edit mode (enter command mode) |

**Cell Management:**

| Shortcut | Action |
|---|---|
| `A` | Insert a cell above (command mode) |
| `B` | Insert a cell below (command mode) |
| `D, D` | Delete the selected cell (press D twice in command mode) |
| `M` | Convert cell to Markdown (command mode) |
| `Y` | Convert cell to Code (command mode) |
| `Z` | Undo cell deletion |
| `Ctrl + Shift + V` | Paste cell above |

### Moving Cells Up or Down

In VS Code notebooks, you can **reorder cells** by:

1. **Drag and drop** — Click the left margin of a cell and drag it to a new position
2. **Toolbar buttons** — Select a cell, then use the **up/down arrow buttons** in the cell toolbar (top-left of each cell)
3. **Keyboard** — VS Code does not have a default keybinding for moving cells, but you can add one:
   - Open Command Palette (`Ctrl + Shift + P`)
   - Search for `Notebook: Move Cell Up` or `Notebook: Move Cell Down`
   - Right-click → "Add Keybinding" (e.g., `Alt + Shift + Up/Down`)

### Other Important Notebook Tips

- **Restart Kernel:** If your notebook is in a bad state (stale variables, import errors), restart the kernel: `Ctrl + Shift + P` → `Notebook: Restart Kernel`
- **Run All Above:** Right-click a cell → "Run All Above" to re-execute everything up to that point
- **Clear Outputs:** `Ctrl + Shift + P` → `Notebook: Clear All Outputs` — useful before committing to Git (output diffs are noisy)
- **Cell execution order matters:** Cells run in the order you execute them, not their position. If you run cell 5 before cell 3, variables from cell 3 will not exist yet. Always use `Run All` to ensure correct order.
- **Variable state persists:** After running a cell, its variables stay in memory until you restart the kernel. This is powerful but can cause confusion if you delete a cell — the variables from that cell still exist in the running kernel.
- **Autosave:** VS Code autosaves notebooks, but outputs are saved too. Use `Clear All Outputs` before Git commits to keep diffs clean.
- **Table of Contents:** Use the Outline panel (`Ctrl + Shift + P` → `Explorer: Focus on Outline View`) to navigate notebooks by markdown headings.

---
## 6. Git & Git Bash — CLI Reference

Git is a **distributed version control system**. Every developer has a full copy of the repository history. Git Bash provides a Unix-like terminal on Windows.

Press "q" anytime to return to kernel prompt

### Initial Setup

```bash
# Configure your identity (one-time)
git config --global user.name "Your Name"
git config --global user.email "you@example.com"

# Set default branch name to 'main'
git config --global init.defaultBranch main

# Enable colored output
git config --global color.ui auto

# Set default editor (e.g., VS Code)
git config --global core.editor "code --wait"
```

---
### Fetching / Cloning a Repository

```bash
# Clone a remote repository (creates a local copy)
git clone https://github.com/user/repo.git

# Clone into a specific folder name
git clone https://github.com/user/repo.git my-folder

# Fetch latest changes from remote (does NOT merge)
git fetch origin

# Pull latest changes (fetch + merge in one step)
git pull origin main

# Pull with rebase (cleaner history — avoids merge commits)
git pull --rebase origin main
```

> **`fetch` vs `pull`:** `fetch` downloads changes but leaves your working directory untouched — safe for inspection. `pull` downloads AND merges — convenient but can cause conflicts. Prefer `fetch` + `merge` when you want to review changes first.

---
### Staging & Committing (Saving Your Work)

Git has a **three-stage workflow:**

```
Working Directory  →  Staging Area  →  Repository
 (your files)         (git add)        (git commit)
```

```bash
# Check the current status (which files are modified, staged, untracked)
git status

# Stage a specific file
git add filename.py

# Stage multiple specific files
git add file1.py file2.py notebook.ipynb

# Stage all changes in the current directory
git add .

# Stage all changes in the entire repo
git add -a

# Unstage a file (remove from staging, keep changes)
git restore --staged filename.py

# Commit staged changes with a message
git commit -m "Add login feature"

# Commit with a multi-line message
git commit -m "Add login feature" -m "Includes form validation and session handling."

# Amend the most recent commit (e.g., fix a typo in the message)
git commit --amend -m "Add login feature with validation"
```

> **Commit message best practices:**
> - Use the imperative mood: "Add feature" not "Added feature"
> - Keep the first line under 72 characters
> - Explain **why**, not just **what** — the diff shows *what* changed

---
### Viewing Commit History

```bash
# Full commit log (press q to quit)
git log

# Compact one-line-per-commit view
git log --oneline

# Show the last 5 commits
git log --oneline -5

# Log with a graphical branch view
git log --oneline --graph --all

# Show details of a specific commit
git show abc1234

# See what changed in each commit (patch view)
git log -p

# See who last modified each line of a file
git blame filename.py

# See changes between your working directory and the last commit
git diff

# See changes that are staged (ready to commit)
git diff --staged

# Compare two branches
git diff main..feature-branch
```

---
### Branching

Branches let you work on features or fixes **in isolation** without affecting the main codebase.

```bash
# List all local branches (* marks the current branch)
git branch

# List all branches including remote
git branch -a

# Create a new branch
git branch feature-login

# Switch to a branch
git checkout feature-login
# OR (modern alternative)
git switch feature-login

# Create AND switch to a new branch in one step
git checkout -b feature-login
# OR
git switch -c feature-login

# Rename the current branch
git branch -m new-name

# Delete a branch (only if fully merged)
git branch -d feature-login

# Force delete an unmerged branch (careful!)
git branch -D feature-login
```

> **Branch naming conventions:**
> - `feature/login-page` — new functionality
> - `bugfix/fix-navbar` — bug fixes
> - `hotfix/security-patch` — urgent production fixes
> - `release/v1.2.0` — release preparation

---
### Merging

Merging combines the work from one branch into another.

```bash
# Step 1: Switch to the branch you want to merge INTO
git checkout main

# Step 2: Merge the feature branch into main
git merge feature-login

# If there are conflicts, Git will pause and mark the files.
# After resolving conflicts manually:
git add resolved-file.py
git commit -m "Merge feature-login into main"

# Abort a merge in progress (go back to pre-merge state)
git merge --abort
```

#### Merge vs Rebase

| Approach | What it does | When to use |
|---|---|---|
| `git merge` | Creates a merge commit joining two histories | Default; preserves full branch history |
| `git rebase main` | Replays your branch commits on top of main | Cleaner linear history; use before merging |

```bash
# Rebase your feature branch onto main (while on feature branch)
git rebase main

# If conflicts occur during rebase:
# Fix the file, then:
git add resolved-file.py
git rebase --continue

# Abort a rebase
git rebase --abort
```

> **Golden rule of rebase:** Never rebase commits that have already been pushed and shared with others. Rebase rewrites history — this causes problems for anyone who based work on the old commits.

---
### Pushing to Remote

```bash
# Push the current branch to the remote
git push origin main

# Push and set upstream (so future pushes just need 'git push')
git push -u origin feature-login

# After setting upstream, simply:
git push

# Push all branches
git push --all origin

# Push tags
git push origin --tags

# Delete a remote branch
git push origin --delete feature-login
```

---
### Stashing (Temporary Storage)

Stash lets you save uncommitted changes temporarily — useful when you need to switch branches but are not ready to commit.

```bash
# Stash current changes
git stash

# Stash with a descriptive message
git stash push -m "WIP: login form styling"

# List all stashes
git stash list

# Apply the most recent stash (keeps it in stash list)
git stash apply

# Apply AND remove the most recent stash
git stash pop

# Apply a specific stash
git stash apply stash@{2}

# Drop a specific stash
git stash drop stash@{0}

# Clear all stashes
git stash clear
```

---
### Undoing Changes

```bash
# Discard changes in a file (revert to last commit)
git restore filename.py

# Unstage a file (keep changes in working directory)
git restore --staged filename.py

# Revert a specific commit (creates a NEW commit that undoes it — safe)
git revert abc1234

# Reset to a previous commit (CAREFUL — rewrites history)
git reset --soft HEAD~1     # Undo last commit, keep changes staged
git reset --mixed HEAD~1    # Undo last commit, keep changes unstaged (default)
git reset --hard HEAD~1     # Undo last commit, DISCARD all changes (dangerous!)
```

> **`revert` vs `reset`:** Use `revert` on shared branches (it is safe — creates a new commit). Use `reset` only on local, unpushed commits.

---
### Tags (Marking Releases)

```bash
# Create a lightweight tag
git tag v1.0.0

# Create an annotated tag (recommended — includes message and metadata)
git tag -a v1.0.0 -m "First stable release"

# List all tags
git tag

# Push a tag to remote
git push origin v1.0.0

# Push all tags
git push origin --tags

# Delete a local tag
git tag -d v1.0.0

# Delete a remote tag
git push origin --delete v1.0.0
```

---
### `.gitignore` — What NOT to Track

Create a `.gitignore` file in your project root to exclude files from version control.

```gitignore
# Virtual environments
.venv/
venv/
env/

# Python cache
__pycache__/
*.pyc
*.pyo

# Jupyter checkpoints
.ipynb_checkpoints/

# IDE settings
.vscode/
.idea/

# OS files
.DS_Store
Thumbs.db

# Environment variables / secrets
.env
*.key
```

> **Tip:** Use https://www.toptal.com/developers/gitignore to generate a `.gitignore` for your tech stack.

---
## Quick Reference — Complete Git Workflow

```bash
# 1. Clone the repo
git clone https://github.com/user/repo.git
cd repo

# 2. Create a feature branch
git switch -c feature/my-feature

# 3. Make changes, then stage and commit
git add .
git commit -m "Add my feature"

# 4. Push the branch to remote
git push -u origin feature/my-feature

# 5. Create a Pull Request on GitHub (or use gh CLI)
gh pr create --title "Add my feature" --body "Description here"

# 6. After PR is approved and merged, clean up
git switch main
git pull origin main
git branch -d feature/my-feature
```

> **Perspective:** Git is the most important tool in a developer's toolkit after the code editor itself. Every professional software team uses Git. The commands above will handle 95% of your daily workflow. Master `status`, `add`, `commit`, `push`, `pull`, `branch`, and `merge` — everything else builds on these fundamentals.